Shotgun Metagenomics（鸟枪法宏基因组）测的是 DNA。

CheckM2: a rapid, scalable and accurate tool for assessing microbial genome quality using machine learning
Nature Methods.2023-07-27;20(8)
CheckM2是一种基于机器学习预测宏基因组组装基因组（MAG）完整度和污染率的改进方法；
2
CheckM2不依赖于谱系特异性标记集，更适合分析代表性较差或新谱系基因组，无需明确分类信息；
3
在已知完整度和污染率的模拟基因组上训练CheckM2，并应用于来自多种不同环境（土壤、肠道和海洋等）的MAG；
4
CheckM2比现有方法更快、更准确，当应用于新的谱系和基因组大小减少的谱系时（如Patescibacteria和DPANN超级门），其性能优于CheckM1和BUSCO。

shotgun meta 分析 详解“Shotgun Meta 分析”通常是指 鸟枪法宏基因组（Shotgun Metagenomics） 数据的分析。
它与常见的 16S rRNA 扩增子分析不同，不针对特定的基因片段，而是将环境样本中所有微生物的 DNA 随机打断成小片段进行测序。
这种方法不仅能告诉你“这里有什么微生物”（物种组成），还能告诉你“它们在做什么”（功能代谢）。


1. 核心分析流程 (Pipeline)通常包含以下几个关键步骤：

质控与预处理 (QC & Preprocessing)：去除低质量序列、接头污染以及宿主 DNA（如人体或植物 DNA）。

物种分类鉴定 (Taxonomic Profiling)：将测序得到的 Reads 与已知数据库比对。常用工具包括 MetaPhlAn（基于标记基因）或 Kraken2（基于 K-mer）。

功能分析 (Functional Profiling)：分析样本中存在的基因功能和代谢途径。常用工具如 HUMAnN。

序列组装 (Assembly)：将短序列拼接成长片段（Contigs）或支架（Scaffolds），常用工具为 MEGAHIT 或 SPAdes。

分箱 (Binning)：根据序列特征（如 GC 含量、丰度等）将组装好的片段归类，重构出单菌水平的基因组: 宏基因组组装基因组（MAG, Metagenome-Assembled Genome）。



2. Shotgun 与 16S 分析的对比
特性               16S rRNA 扩增子               Shotgun 宏基因组
检测范围           仅细菌/古菌（特定引物）         细菌、古菌、病毒、真菌
分辨率             通常到“属”水平                 可达“株”水平 (Strain level)
功能预测           基于已知物种的推测 (PICRUSt)    直接检测基因，真实的功能分析
成本               较低                           较高


3. 常用资源与工具综合流程：
bioBakery 是一套非常流行的 shotgun 分析工具集。
数据库：常用的包括 NCBI RefSeq、GTDB（物种）和 KEGG、MetaCyc（功能）

### Taxonomic Profiling: MetaPhlAn VS Kraken2

| 对比维度 | MetaPhlAn (当前版本 4) | Kraken2 |
| :--- | :--- | :--- |
| **底层算法** | **基于特异性标记基因 (Marker-gene)**。只抽取细菌基因组中高度保守且特异的单拷贝基因片段进行比对。 | **基于精确 K-mer 匹配 (K-mer based)**。将 Read 拆分成 k-mers，通过 LCA (最低共同祖先) 算法在整个基因组数据库中快速查找匹配。 |
| **计算资源** | 极低。普通笔记本或小服务器即可运行。 | **极高**。需要庞大的内存来加载哈希表（标准库需要 50GB 以上内存，定制超大库可能需要几百 GB）。 |
| **运行速度** | 较快（主要进行 Bowtie2 比对）。 | **极快**（目前最快的注释工具之一）。 |
| **假阳性与假阴性** | 假阳性**极低**（非常严谨）。但如果样本中存在数据库中完全没有的新物种（缺乏 marker），则会产生假阴性。 | 假阳性**较高**。容易将相近物种的同源序列误判，需要严格的阈值过滤。 |
| **输出结果** | 直接输出物种的**相对丰度表**（Relative abundance）。 | 默认输出基于 Read 数量的分类结果。**强烈建议配合 Bracken 软件使用**，以重新估算真实的物种相对丰度。 |

### Assembly: MEGAHIT VS SPAdes

| 对比维度 | MEGAHIT | SPAdes (metaSPAdes) |
| :--- | :--- | :--- |
| **底层算法** | 简洁德布鲁因图 (Succinct de Bruijn graphs)。 | 多 K-mer 迭代的 de Bruijn graphs。 |
| **资源消耗** | **极度节省内存**。算法经过高度优化，处理几百 GB 的复杂土壤宏基因组数据也能游刃有余。 | **极度消耗内存**。运算过程会产生大量中间文件，对于复杂样本（如土壤环境），经常会导致服务器 OOM（内存溢出）崩溃。 |
| **拼接质量** | N50 表现良好，擅长拼接复杂的、高丰度差异的群落。但在极其相似的菌株分辨率上稍逊一筹。 | **拼接质量极高**。N50 通常更长，错误率极低，能够更精细地解析由于测序深度不均带来的问题。 |
| **使用建议** | 绝大多数环境样本、超大样本量、或者**内存有限**（如你的个人老旧服务器）的**首选**。 | 如果服务器内存足够（几百 GB 以上），且样本相对简单（如单个人类肠道样本），追求极致质量时选用。 |

Bowtie2 和 BWA-MEM： 它们是**非剪接感知（Non-splice-aware）**的。遇到这种跨越内含子的 read，它们会直接判定为“比对失败”或强行错配。

真正的 RNA-seq 软件： 必须是**剪接感知（Splice-aware）**的。比如 STAR 或者 HISAT2。它们内部有特殊的算法来处理跨越内含子的“劈叉”比对。